# 10-04 Agent 策略模式

**核心问题**: 不同任务需要不同的 Agent 推理策略，选错策略 = 浪费 token + 效果差。

**本节目标**：ReAct、Plan-and-Execute、Reflection、Tool 选择策略

---

In [ ]:
# ReAct: Reasoning + Acting 交替执行
import json

class ReActAgent:
    """ReAct Agent: 思考→行动→观察 循环"""
    
    def __init__(self, tools: dict):
        self.tools = tools
        self.trajectory = []
    
    def think(self, observation: str) -> dict:
        """模拟 LLM 思考（实际应调用 LLM）"""
        # 简化: 基于关键词决策
        if '天气' in observation:
            return {"thought": "用户问天气，需要调用天气工具", "action": "weather", "action_input": "北京"}
        elif '计算' in observation or any(c.isdigit() for c in observation):
            return {"thought": "需要计算，调用计算器", "action": "calculator", "action_input": observation}
        elif '搜索' in observation or '什么是' in observation:
            return {"thought": "需要搜索信息", "action": "search", "action_input": observation}
        return {"thought": "可以直接回答", "action": "finish", "action_input": observation}
    
    def act(self, action: str, action_input: str) -> str:
        if action == "finish":
            return action_input
        tool = self.tools.get(action)
        if tool:
            return tool(action_input)
        return f"工具 {action} 不存在"
    
    def run(self, query: str, max_steps: int = 5) -> str:
        observation = query
        for step in range(max_steps):
            # Thought
            result = self.think(observation)
            self.trajectory.append({"step": step + 1, "type": "think", **result})
            print(f"  Step {step+1} Thought: {result['thought']}")
            
            # Act
            if result["action"] == "finish":
                print(f"  Step {step+1} Final: {result['action_input']}")
                return result["action_input"]
            
            observation = self.act(result["action"], result["action_input"])
            self.trajectory.append({"step": step + 1, "type": "observe", "observation": observation})
            print(f"  Step {step+1} Action: {result['action']}({result['action_input']})")
            print(f"  Step {step+1} Observation: {observation}")
        
        return "达到最大步数限制"

# 定义工具
tools = {
    "weather": lambda city: f"{city}今天晴，25°C",
    "calculator": lambda expr: f"计算结果: 42",
    "search": lambda q: f"搜索结果: Agent 是能自主决策和执行任务的 AI 系统",
}

agent = ReActAgent(tools)
print("=== ReAct 示例 1: 天气查询 ===")
agent.run("北京天气怎么样？")

print("\n=== ReAct 示例 2: 知识搜索 ===")
agent2 = ReActAgent(tools)
agent2.run("什么是 Agent？")

print("""
ReAct 核心思想:
  Thought → Action → Observation → Thought → ... → Final Answer
  
优点: 推理过程可解释，适合需要工具调用的任务
缺点: 每步都要调 LLM，token 消耗大；复杂任务容易迷失
""")

In [ ]:
# Plan-and-Execute: 先规划，再逐步执行
class PlanAndExecuteAgent:
    """先制定计划，再逐步执行"""
    
    def plan(self, goal: str) -> list[str]:
        """制定执行计划（实际应调用 LLM）"""
        # 模拟 LLM 规划
        plans = {
            "分析竞品广告策略": [
                "1. 收集竞品广告数据（CTR/CVR/花费）",
                "2. 分析各竞品的投放时段和人群定向",
                "3. 对比创意风格和文案特点",
                "4. 生成分析报告和优化建议",
            ],
            "优化广告投放ROI": [
                "1. 拉取最近7天投放数据",
                "2. 识别低效广告计划（CPA>目标值）",
                "3. 分析低效原因（创意/定向/出价）",
                "4. 生成优化方案并执行",
            ],
        }
        for key in plans:
            if key in goal:
                return plans[key]
        return ["1. 理解任务需求", "2. 收集相关信息", "3. 执行核心操作", "4. 验证结果"]
    
    def execute_step(self, step: str, context: str) -> str:
        """执行单步（实际应调用 LLM + Tools）"""
        return f"✅ 完成: {step} (基于上下文: {context[:30]}...)"
    
    def replan(self, remaining_steps: list, feedback: str) -> list:
        """根据执行反馈调整计划"""
        print(f"  [Replan] 根据反馈调整: {feedback[:50]}")
        return remaining_steps  # 简化: 保持原计划
    
    def run(self, goal: str) -> str:
        print(f"目标: {goal}\n")
        
        # Phase 1: Plan
        steps = self.plan(goal)
        print("=== 计划 ===")
        for s in steps:
            print(f"  {s}")
        
        # Phase 2: Execute
        print("\n=== 执行 ===")
        context = goal
        results = []
        for i, step in enumerate(steps):
            result = self.execute_step(step, context)
            results.append(result)
            context += f"\n{result}"
            print(f"  {result}")
        
        summary = f"完成 {len(results)}/{len(steps)} 步"
        print(f"\n=== 结果: {summary} ===")
        return summary

agent = PlanAndExecuteAgent()
agent.run("分析竞品广告策略")

print("""
Plan-and-Execute vs ReAct:
  
  Plan-and-Execute:
    - 先全局规划，再逐步执行
    - 适合: 复杂多步任务、需要全局视角
    - 优点: 结构清晰，可 replan
    - 缺点: 初始规划可能不准确
  
  ReAct:
    - 边思考边执行，逐步推进
    - 适合: 简单查询、需要实时反馈
    - 优点: 灵活，能快速响应
    - 缺点: 缺乏全局视角，复杂任务易迷失
""")

In [ ]:
# Reflection: 自我反思 + 改进
class ReflectionAgent:
    """通过自我反思不断改进输出"""
    
    def generate(self, task: str, feedback: str = "") -> str:
        """生成初始/改进版本"""
        if not feedback:
            return "B站大会员年卡限时优惠，追番无广告，专属内容畅享！"
        # 根据反思改进
        return "追番党福利！大会员年卡限时特惠，1080P无广告+专属番剧，快来体验！"
    
    def reflect(self, output: str, criteria: list[str]) -> dict:
        """自我反思：评估输出质量"""
        issues = []
        suggestions = []
        score = 8.0
        
        if len(output) > 30:
            issues.append("文案偏长，可以更精炼")
            score -= 1
        if '！' not in output:
            issues.append("缺少感叹号，不够有感染力")
            score -= 0.5
        if not any(w in output for w in ['限时', '专属', '福利', '畅享']):
            issues.append("缺少紧迫感/独特性词汇")
            score -= 1
        
        for c in criteria:
            if c == "吸引力" and '！' in output:
                suggestions.append("感叹号使用得当")
            if c == "简洁性" and len(output) <= 25:
                suggestions.append("长度合适")
        
        return {
            "score": score,
            "issues": issues,
            "suggestions": suggestions,
            "needs_improvement": score < 7.5
        }
    
    def run(self, task: str, criteria: list[str], max_iterations: int = 3) -> str:
        print(f"任务: {task}\n评估标准: {criteria}\n")
        
        output = self.generate(task)
        for i in range(max_iterations):
            print(f"--- 第 {i+1} 轮 ---")
            print(f"  输出: {output}")
            
            reflection = self.reflect(output, criteria)
            print(f"  评分: {reflection['score']}")
            print(f"  问题: {reflection['issues']}")
            
            if not reflection['needs_improvement']:
                print(f"  ✅ 质量达标，停止迭代")
                break
            
            output = self.generate(task, feedback=str(reflection['issues']))
            print(f"  → 改进后重新生成")
        
        return output

agent = ReflectionAgent()
result = agent.run(
    task="为B站大会员写广告文案",
    criteria=["吸引力", "简洁性", "合规性"]
)

print(f"\n最终输出: {result}")

print("""
Reflection 模式关键点:
  1. Generate → Reflect → Improve 循环
  2. 反思时需要明确的评估标准 (criteria)
  3. 设置最大迭代次数防止死循环
  4. 适用场景: 文案生成、代码生成、方案设计等需要打磨的任务
  
变体:
  - Reflexion: 加入外部反馈 (如执行结果)
  - Self-Refine: 纯 LLM 自我改进
  - CRITIC: 用工具验证 LLM 输出
""")

In [ ]:
# Tool 选择策略
class ToolSelector:
    """智能工具选择器"""
    
    def __init__(self):
        self.tools = {
            "ad_data_query": {
                "description": "查询广告投放数据（CTR/CVR/花费/曝光）",
                "keywords": ["数据", "CTR", "CVR", "花费", "曝光", "报表", "查询"],
                "cost": "low",
            },
            "creative_generator": {
                "description": "生成广告创意文案",
                "keywords": ["生成", "文案", "创意", "标题", "素材"],
                "cost": "medium",
            },
            "audience_analyzer": {
                "description": "分析目标受众画像",
                "keywords": ["受众", "人群", "画像", "用户", "定向"],
                "cost": "medium",
            },
            "budget_optimizer": {
                "description": "优化预算分配方案",
                "keywords": ["预算", "分配", "优化", "ROI", "出价"],
                "cost": "high",
            },
        }
    
    def select(self, query: str, strategy: str = "keyword") -> list[str]:
        """选择合适的工具"""
        if strategy == "keyword":
            return self._keyword_match(query)
        elif strategy == "cost_aware":
            return self._cost_aware_select(query)
        return list(self.tools.keys())[:1]
    
    def _keyword_match(self, query: str) -> list[str]:
        scores = {}
        for name, tool in self.tools.items():
            score = sum(1 for kw in tool["keywords"] if kw in query)
            if score > 0:
                scores[name] = score
        return sorted(scores, key=scores.get, reverse=True)
    
    def _cost_aware_select(self, query: str) -> list[str]:
        """成本感知选择: 优先选低成本工具"""
        cost_order = {"low": 0, "medium": 1, "high": 2}
        candidates = self._keyword_match(query)
        return sorted(candidates, key=lambda t: cost_order.get(self.tools[t]["cost"], 99))

selector = ToolSelector()

queries = [
    "查询昨天的广告CTR数据",
    "生成一个游戏皮肤的广告文案",
    "分析18-25岁用户画像并优化预算分配",
]

for q in queries:
    kw_result = selector.select(q, "keyword")
    cost_result = selector.select(q, "cost_aware")
    print(f"查询: {q}")
    print(f"  关键词匹配: {kw_result}")
    print(f"  成本优先:   {cost_result}")
    print()

print("""
工具选择策略对比:
  1. 关键词匹配: 简单快速，适合工具少的场景
  2. Embedding 语义匹配: 更准确，适合工具多（>20个）
  3. LLM 选择: 最灵活，但增加一次 LLM 调用
  4. 成本感知: 优先选低成本工具，控制 API 开销
  5. 级联选择: 先用便宜的，不够再用贵的
""")

In [ ]:
# 策略选择指南
print("""
=== Agent 策略选择决策树 ===

任务来了 →
  ├─ 简单查询/单步操作 → 直接 LLM 调用（无需 Agent）
  ├─ 需要工具调用 → ReAct
  │   └─ 工具 < 5个? → 直接 ReAct
  │   └─ 工具 > 5个? → 先 Tool Selection → 再 ReAct
  ├─ 复杂多步任务 → Plan-and-Execute
  │   └─ 步骤间强依赖? → 串行执行
  │   └─ 步骤可并行? → 并行执行（Map-Reduce）
  ├─ 需要高质量输出 → Reflection
  │   └─ 有外部验证? → Reflexion（代码执行/测试）
  │   └─ 纯文本输出? → Self-Refine
  └─ 多角色协作 → Multi-Agent
      └─ 固定流程? → Pipeline 模式
      └─ 动态分工? → Supervisor 模式

=== B站广告场景策略映射 ===

  场景1: "查询昨天CTR" → 直接工具调用（无需 Agent）
  场景2: "分析数据并生成优化建议" → ReAct（查询→分析→建议）
  场景3: "制定完整投放方案" → Plan-and-Execute（规划→执行→验证）
  场景4: "生成高质量广告文案" → Reflection（生成→审核→改进）
  场景5: "端到端投放优化" → Multi-Agent（分析+策略+创意+监控）
""")

## 面试速记

| 问题 | 要点 |
|------|------|
| ReAct 原理 | Thought→Action→Observation 循环；推理与行动交替；可解释性强 |
| Plan-and-Execute vs ReAct | P&E 先全局规划再执行，适合复杂任务；ReAct 边想边做，适合简单查询 |
| Reflection 怎么用 | Generate→Reflect→Improve 循环；需要明确评估标准；设最大迭代数 |
| 如何选择策略 | 简单→直接调用；需工具→ReAct；多步复杂→P&E；高质量→Reflection；多角色→Multi-Agent |
| Tool 选择策略 | 关键词匹配(简单)、Embedding语义(准确)、LLM选择(灵活)、成本感知(省钱) |